# Assignment 2: training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file. `main.py` refuses to run if
it finds both `train.py` and `train.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py train`, this notebook must, start to finish, with no
manual steps in between:

- Get your Roboflow API key via `from api_key import load_api_key`, so that it
  works whether the key is in your `.env` file or in the environment, and
  download your dataset version from Roboflow Universe.
- Finetune yolo11n/yolo11s (or the -cls variant) on that dataset, from the
  pretrained checkpoint.
- Evaluate on the test split against the two baselines named in the brief: the
  pretrained COCO model zero-shot, and a trivial predict-nothing / majority
  class baseline.
- Sweep the confidence threshold, compute the expected cost at each point from
  the cost matrix you defined in REPORT.md, and save the sweep.
- Save everything `app.py` needs to build its dashboard without retraining:
  the metrics, the per-class numbers, the cost sweep, and the chosen operating
  point.

This is your real training pipeline. It takes as long as it takes, and it runs
wherever training is practical for you: your own machine, a GPU you have access
to, or a hosted notebook. Nothing re-runs it to check your submission.

The weights it produces are what you commit to `weights/` and what your app
serves. Say in REPORT.md where you trained and what settings you used.

Save the trained model into `weights/`. That is the file you commit and the
file your app loads.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.


In [ ]:
import json
import os
import shutil
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

import costing

WORKSPACE = "medical-object-detection-zblyv"
PROJECT = "patient-fall-detection"
VERSION = 11
DATA_DIR = Path("datasets") / f"{PROJECT}-v{VERSION}"

POSITIVE_CLASSES = ["fallen", "falling"]

BASE_MODEL = "yolo11n.pt"
EPOCHS = int(os.environ.get("EPOCHS", 50))
FRACTION = float(os.environ.get("FRACTION", 1.0))
IMGSZ = 640
BATCH = 16
SEED = 0
DEVICE = os.environ.get("DEVICE") or ("0" if torch.cuda.is_available() else "cpu")

WEIGHTS_DIR = Path("weights")
ARTIFACTS = Path("artifacts")
WEIGHTS_DIR.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)

IMAGES_PER_FALL = 60
INCIDENT_COSTS = {"fn": 25000, "tp": 3}
COSTS = {
    "fn": INCIDENT_COSTS["fn"] / IMAGES_PER_FALL,
    "fp": 0.43,
    "tp": INCIDENT_COSTS["tp"] / IMAGES_PER_FALL,
    "tn": 0,
    "review": 0.05,
}
BASE_RATE = 0.0000057
COSTS_SET = BASE_RATE is not None and all(v is not None for v in COSTS.values())
print("Device:", DEVICE, "| epochs:", EPOCHS, "| fraction:", FRACTION, "| costs set:", COSTS_SET)

In [ ]:
if not (DATA_DIR / "data.yaml").exists():
    from roboflow import Roboflow
    from api_key import load_api_key

    rf = Roboflow(api_key=load_api_key())
    rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download(
        "yolov11", location=str(DATA_DIR)
    )

names = yaml.safe_load((DATA_DIR / "data.yaml").read_text())["names"]
names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
POSITIVE_IDS = [names.index(c) for c in POSITIVE_CLASSES]

DATA_YAML = DATA_DIR / "data_abs.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": dict(enumerate(names)),
}))
print("Classes:", names, "| positive:", POSITIVE_CLASSES)

In [ ]:
def read_labels(split):
    rows = {}
    for f in sorted((DATA_DIR / split / "labels").glob("*.txt")):
        boxes = []
        for line in f.read_text().split("\n"):
            p = line.split()
            if len(p) >= 5:
                boxes.append((int(p[0]), float(p[3]) * float(p[4])))
        rows[f.stem] = boxes
    return rows


def fingerprint(path):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    small = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA)
    return (small[:, 1:] > small[:, :-1]).tobytes()


dataset_stats = {"splits": {}}
for split in ["train", "valid", "test"]:
    labels = read_labels(split)
    all_boxes = [b for boxes in labels.values() for b in boxes]
    dataset_stats["splits"][split] = {
        "images": len(labels),
        "images_with_fall": sum(any(c in POSITIVE_IDS for c, _ in b) for b in labels.values()),
        "median_box_share": float(np.median([a for _, a in all_boxes])),
        "boxes_per_class": {n: sum(c == i for c, _ in all_boxes) for i, n in enumerate(names)},
        "images_per_class": {n: sum(any(c == i for c, _ in b) for b in labels.values())
                             for i, n in enumerate(names)},
    }

train_prints = {fingerprint(p) for p in (DATA_DIR / "train" / "images").glob("*")}
test_images = sorted((DATA_DIR / "test" / "images").glob("*"))
dataset_stats["test_images_matching_a_train_fingerprint"] = sum(
    fingerprint(p) in train_prints for p in test_images
)

print(pd.DataFrame({s: v["boxes_per_class"] for s, v in dataset_stats["splits"].items()}))
for s, v in dataset_stats["splits"].items():
    print(f"{s}: {v['images']} images, {v['images_with_fall']} with a fall, "
          f"median box {v['median_box_share']:.1%} of image")
print("Test images matching a train fingerprint:",
      dataset_stats["test_images_matching_a_train_fingerprint"], "of", len(test_images))

In [ ]:
SKIP_TRAIN = os.environ.get("SKIP_TRAIN") == "1" and (WEIGHTS_DIR / "best.pt").exists()
if not SKIP_TRAIN:
    model = YOLO(BASE_MODEL)
    model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, fraction=FRACTION,
        device=DEVICE, seed=SEED, project="runs", name="train", exist_ok=True, plots=True,
    )
    best = Path(model.trainer.save_dir) / "weights" / "best.pt"
    shutil.copy(best, WEIGHTS_DIR / "best.pt")
model = YOLO(WEIGHTS_DIR / "best.pt")
print("Using", WEIGHTS_DIR / "best.pt", "| retrained:", not SKIP_TRAIN)

In [ ]:
CM_CONF = 0.25

test_metrics = model.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, device=DEVICE,
                         project="runs", name="test", exist_ok=True, plots=True)
cm_metrics = model.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, device=DEVICE,
                       conf=CM_CONF, project="runs", name="test_cm", exist_ok=True, plots=True)

per_class = pd.DataFrame(test_metrics.summary())
labels = names + ["background"]
detection_results = {
    "mAP50": float(test_metrics.box.map50),
    "mAP50-95": float(test_metrics.box.map),
    "per_class": per_class.to_dict(orient="records"),
    "confusion_conf": CM_CONF,
    "confusion_matrix": cm_metrics.confusion_matrix.matrix.astype(int).tolist(),
    "confusion_labels": labels,
}
shutil.copy(Path(cm_metrics.save_dir) / "confusion_matrix.png", ARTIFACTS / "confusion_matrix.png")
print(per_class.to_string(index=False))
print(f"mAP50 = {detection_results['mAP50']:.3f}, mAP50-95 = {detection_results['mAP50-95']:.3f}")
print(pd.DataFrame(detection_results["confusion_matrix"],
                   index=[f"pred {n}" for n in labels], columns=[f"true {n}" for n in labels]))

In [ ]:
coco = YOLO(BASE_MODEL)
rows = []
for img in test_images:
    label_file = DATA_DIR / "test" / "labels" / f"{img.stem}.txt"
    true_ids = {int(l.split()[0]) for l in label_file.read_text().split("\n") if l.strip()}

    ft = model.predict(img, conf=0.01, imgsz=IMGSZ, device=DEVICE, verbose=False)[0].boxes
    ft_scores = [float(c) for c, k in zip(ft.conf, ft.cls) if int(k) in POSITIVE_IDS]

    cz = coco.predict(img, conf=0.01, classes=[0], imgsz=IMGSZ, device=DEVICE, verbose=False)[0].boxes
    cz_scores = [float(c) for c, (w, h) in zip(cz.conf, cz.xywh[:, 2:].tolist()) if w > h]

    rows.append({
        "image": img.name,
        "is_fall": bool(true_ids & set(POSITIVE_IDS)),
        "score_finetuned": max(ft_scores, default=0.0),
        "score_coco": max(cz_scores, default=0.0),
        "score_trivial": 0.0,
    })

scores = pd.DataFrame(rows)
scores.to_csv(ARTIFACTS / "test_scores.csv", index=False)
print(scores["is_fall"].value_counts().rename({True: "fall", False: "no fall"}))

In [ ]:
def average_precision(score, y):
    order = np.argsort(-score)
    y = y[order]
    if y.sum() == 0:
        return 0.0
    hits = np.cumsum(y)
    precision = hits / np.arange(1, len(y) + 1)
    return float((precision * y).sum() / y.sum())


y = scores["is_fall"].to_numpy()
baselines = {}
for name in ["finetuned", "coco", "trivial"]:
    s = scores[f"score_{name}"].to_numpy()
    cm = costing.confusion_at(s, y, 0.5)
    baselines[name] = {
        **cm,
        "recall": cm["tp"] / max(cm["tp"] + cm["fn"], 1),
        "precision": cm["tp"] / max(cm["tp"] + cm["fp"], 1),
        "average_precision": average_precision(s, y) if name != "trivial" else None,
    }
print(pd.DataFrame(baselines).T.round(3))

In [ ]:
thresholds = np.round(np.arange(0.0, 1.0001, 0.01), 2)
s = scores["score_finetuned"].to_numpy()
sweep = pd.DataFrame([{"threshold": t, **costing.confusion_at(s, y, t)} for t in thresholds])
operating_point = None

if COSTS_SET:
    sweep["cost_per_1000"] = [
        costing.cost_per_1000(s, y, t, t, COSTS, BASE_RATE)[0] for t in thresholds
    ]
    best_t = float(sweep.loc[sweep["cost_per_1000"].idxmin(), "threshold"])

    band = []
    for low in thresholds:
        for high in thresholds[thresholds >= low]:
            cost, review = costing.cost_per_1000(s, y, low, high, COSTS, BASE_RATE)
            band.append((float(cost), float(low), float(high), float(review)))
    band_cost, low, high, review_share = min(band)

    operating_point = {
        "single_threshold": best_t,
        "single_threshold_cost_per_1000": float(sweep["cost_per_1000"].min()),
        "review_low": low,
        "review_high": high,
        "band_cost_per_1000": band_cost,
        "review_share": review_share,
    }

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sweep["threshold"], sweep["cost_per_1000"])
    ax.axvline(best_t, color="red", linestyle="--", label=f"chosen = {best_t:.2f}")
    ax.set_xlabel("Confidence threshold")
    ax.set_ylabel("Expected cost per 1,000 cases (EUR)")
    ax.set_title("Cost against threshold (fine-tuned model)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(ARTIFACTS / "cost_curve.png", dpi=150)
    print(operating_point)
else:
    print("Costs not set yet: saved the threshold sweep counts, skipped the cost curve.")

sweep.to_csv(ARTIFACTS / "threshold_sweep.csv", index=False)

In [ ]:
alternatives = None
sensitivity = None
if COSTS_SET:
    all_pass = 1.01
    alternatives = {
        "do_nothing": costing.cost_per_1000(s, y, all_pass, all_pass, COSTS, BASE_RATE)[0],
        "human_checks_everything": costing.cost_per_1000(s, y, 0.0, all_pass, COSTS, BASE_RATE)[0],
        "model_alone": operating_point["single_threshold_cost_per_1000"],
        "model_plus_review": operating_point["band_cost_per_1000"],
    }
    alternatives = {k: float(v) for k, v in alternatives.items()}

    sensitivity = []
    for p in [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1]:
        costs_at_p = [costing.cost_per_1000(s, y, t, t, COSTS, p)[0] for t in thresholds]
        i = int(np.argmin(costs_at_p))
        sensitivity.append({"base_rate": p, "best_threshold": float(thresholds[i]),
                            "cost_per_1000": float(costs_at_p[i])})

    print(pd.Series(alternatives, name="EUR per 1,000 cases").round(2))
    print(pd.DataFrame(sensitivity).to_string(index=False))

In [ ]:
summary = {
    "dataset": {"workspace": WORKSPACE, "project": PROJECT, "version": VERSION,
                "classes": names, "positive_classes": POSITIVE_CLASSES},
    "training": {"base_model": BASE_MODEL, "epochs": EPOCHS, "fraction": FRACTION,
                 "imgsz": IMGSZ, "batch": BATCH, "seed": SEED, "device": DEVICE},
    "dataset_stats": dataset_stats,
    "detection": detection_results,
    "baselines": baselines,
    "costs": COSTS,
    "incident_costs": INCIDENT_COSTS,
    "images_per_fall": IMAGES_PER_FALL,
    "base_rate": BASE_RATE,
    "operating_point": operating_point,
    "alternatives": alternatives,
    "sensitivity": sensitivity,
}
(ARTIFACTS / "results.json").write_text(json.dumps(summary, indent=2, default=float))
print("Saved:", sorted(p.name for p in ARTIFACTS.iterdir()))